In [1]:
import pathlib
import random
import copy
import numpy as np
import torch
#import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from scipy.fftpack import fft, ifft
from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

CUDA extension for structured kernels (Cauchy and Vandermonde multiplication) not found. Install by going to extensions/kernels/ and running `python setup.py install`, for improved speed and memory efficiency. Note that the kernel changed for state-spaces 4.0 and must be recompiled.


In [2]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [3]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 29
 test_subject_indices: [2,13,24,26,27,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def shift_phase(signal, sampling_rate, low_freq, high_freq, phase_shift=np.pi/4):

    X = fft(signal)
    frequencies = np.fft.fftfreq(len(signal), d=1/sampling_rate)
    band_indices = np.where((frequencies >= low_freq) & (frequencies <= high_freq))

    X[band_indices] *= np.exp(1j * phase_shift)  
    X[-band_indices[0]] = np.conj(X[band_indices])  

    # Reconstruct the modified signal via inverse FFT
    modified_signal = ifft(X).real

    return modified_signal

In [6]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data/model_checkpoints/finetune"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_finetune_subject_index_{subject_index}_start_idx_{start_index}_rep_0_pen.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [7]:
freq_bands = {"delta" : (0.5, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
#amplification_factors = [0.2,0.5,2,3,5,10]
phase_peturbations = np.deg2rad(np.arange(45, 316, 45))

In [8]:
def create_perturbed_samples_dict(all_epochs, freq_bands, phase_shifts):
    perturbed_samples_dict = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        for phase_shift in phase_shifts:
            degree = np.rad2deg(phase_shift)
            perturbed_samples = np.zeros_like(all_epochs)
            for sample in range(all_epochs.shape[0]):
                for ch in range(all_epochs.shape[1]):
                    perturbed_samples[sample, ch] = shift_phase(all_epochs[sample, ch], 1000, low_freq, high_freq, phase_shift)
            perturbed_samples_dict[f"{band_name}_{int(degree)}°"] = perturbed_samples

    return perturbed_samples_dict


In [9]:
def get_predictions(dataset, start_index=100, subject_index=2):
    cfg = load_config()
    pred_label = np.zeros((dataset.shape[0]))
    uncertainties = np.zeros((dataset.shape[0]))
    input_shape_st = (60, 900)                
    for i in range(len(dataset)):
        current_start_index = i + start_index
        inputs = torch.from_numpy(dataset[i])
        inputs = inputs.to(device).float()
        inputs = inputs.unsqueeze(0)

        model = load_model(cfg, start_index=current_start_index, subject_index=subject_index)
        pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
        var = torch.exp(log_var)
        pred_label[i] = pred_mean.cpu().detach().numpy()
        uncertainties[i] = var.cpu().detach().numpy()
    
    return pred_label, uncertainties

In [10]:
def compare_predictions(freq_bands, phase_shifts, predictions_perturbed, pred_label_original):
    #comparison_df = pd.DataFrame(index=amplification_factors, columns=freq_bands.keys())
    for band_name in freq_bands.keys():
        for factor in phase_shifts:
            key = f"{band_name}_{factor}"
            if key in predictions_perturbed:
                original_predictions = pred_label_original
                perturbed_predictions = predictions_perturbed[key]
                median_abs_diff = np.median(original_predictions - perturbed_predictions)
                #comparison_df.loc[factor, band_name] = mean_abs_diff
    #return comparison_df
    return median_abs_diff

In [11]:
input_shape_st = (60, 900)

In [39]:
def generate_perturbed_predictions(all_epochs, freq_bands, phase_peturbations, ch_names, subject_index, device, cfg):
    perturbed_prediction_dict = {}
    for sample in range(all_epochs.shape[0]):
    
        for phase_shift in phase_peturbations:
            perturbed_channel_wise = {}
            for ch_idx, ch_name in enumerate(ch_names):
                pred_label_perturbed = np.zeros((all_epochs.shape[0]))
                uncertainties_perturbed = np.zeros((all_epochs.shape[0]))
                for band_name, (low_freq, high_freq) in tqdm(freq_bands.items()):
                    perturbed_sample = copy.deepcopy(all_epochs[sample])
                    perturbed_sample[ch_idx] = shift_phase(all_epochs[sample, ch_idx], 1000, low_freq, high_freq, phase_shift)
                    inputs = torch.from_numpy(perturbed_sample).to(device).float().unsqueeze(0)
                    print(f"Band {band_name}, phase shift {phase_shift}, Channel {ch_name}, Sample {sample}")
                    print(np.average(np.abs(all_epochs[sample, ch_idx] - perturbed_sample[ch_idx])))
            #        pred_label_perturbed[sample] = pred_mean.cpu().numpy()
            #        uncertainties_perturbed[sample] = var.cpu().numpy()

            #    perturbed_channel_wise[ch_name] = (pred_label_perturbed, uncertainties_perturbed)
            #perturbed_prediction_dict[f"band_{band_name}_phase_shift_{np.rad2deg(phase_shift)}°"] = perturbed_channel_wise
            #np.save(f"perturbed_prediction_dict_{band_name}_channel_wise_phase_shift_{int(np.rad2deg(phase_shift))}°_subject_{subject_index}_new2.npy", perturbed_channel_wise)


In [ ]:
all_subjects_perturbed_predictions = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
    all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:]
    labels_raw = labels_raw[150:]
    print("all epochs shape", all_epochs.shape)
    
    generate_perturbed_predictions(all_epochs, freq_bands, phase_peturbations, ch_names, subject_index, device, cfg)
    #np.save(f"perturbed_predictions_subject_{subject_index}_channel_wise.npy", perturbed_predictions)

# Save the results
#np.save("all_subjects_perturbed_predictions.npy", all_subjects_perturbed_predictions)

Loading EEG data...
(100, 60, 900)
(1, 60, 1)
all epochs shape (571, 60, 900)


100%|██████████| 5/5 [00:00<00:00, 209.27it/s]


Band delta, phase shift 0.7853981633974483, Channel Fp1, Sample 0
0.3640112829807961
Band theta, phase shift 0.7853981633974483, Channel Fp1, Sample 0
0.40558811784718085
Band alpha, phase shift 0.7853981633974483, Channel Fp1, Sample 0
0.29301846168197293
Band beta, phase shift 0.7853981633974483, Channel Fp1, Sample 0
0.30033888030405576
Band gamma, phase shift 0.7853981633974483, Channel Fp1, Sample 0
0.2681144048770992


100%|██████████| 5/5 [00:00<00:00, 1919.59it/s]


Band delta, phase shift 0.7853981633974483, Channel Fp2, Sample 0
0.22169245160052628
Band theta, phase shift 0.7853981633974483, Channel Fp2, Sample 0
0.279874855877927
Band alpha, phase shift 0.7853981633974483, Channel Fp2, Sample 0
0.28703699358803264
Band beta, phase shift 0.7853981633974483, Channel Fp2, Sample 0
0.3181782500930524
Band gamma, phase shift 0.7853981633974483, Channel Fp2, Sample 0
0.39137139915856767


...

Band delta, phase shift 3.141592653589793, Channel CP6, Sample 9
0.8562274164842761
Band theta, phase shift 3.141592653589793, Channel CP6, Sample 9
0.42665904798347315
Band alpha, phase shift 3.141592653589793, Channel CP6, Sample 9
1.364203480936586
Band beta, phase shift 3.141592653589793, Channel CP6, Sample 9
0.9705955062721235


100%|██████████| 5/5 [00:00<00:00, 640.00it/s]


Band gamma, phase shift 3.141592653589793, Channel CP6, Sample 9
0.6664708663934011


100%|██████████| 5/5 [00:00<00:00, 1616.55it/s]

Band delta, phase shift 3.141592653589793, Channel F1, Sample 9
0.5918913583911732
Band theta, phase shift 3.141592653589793, Channel F1, Sample 9
0.9049042931446704
Band alpha, phase shift 3.141592653589793, Channel F1, Sample 9
0.8346981599961588
Band beta, phase shift 3.141592653589793, Channel F1, Sample 9
1.2403354188239464
Band gamma, phase shift 3.141592653589793, Channel F1, Sample 9
0.5167437916822607
